In [6]:
%use kandy

# Parametros

In [5]:
val Tmax = 125
val Tamb = 40

val Rth_jc_igbt     = 0.88
val Rth_jc_diodo    = 1.78
val Rth_cd          = 0.2
val Rth_da          = 0.9
val Pmax_igbt = (Tmax - Tamb) / (Rth_jc_igbt + Rth_cd + Rth_da)
val Pmax_diodo = (Tmax - Tamb) / (Rth_jc_diodo + Rth_cd + Rth_da)

println("Potencia Maxima IGBT: ${String.format("%.2f", Pmax_igbt)} W")
println("Potencia Maxima Diodo: ${String.format("%.2f", Pmax_diodo)} W")

Potencia Maxima IGBT: 42,93 W
Potencia Maxima Diodo: 29,51 W


In [17]:
val Vce_sat = 1.55f //V
val Vce_bloq= 300   //V
val Vce0    = 0.9f    //V
val rce     = (Vce_sat - Vce0) / 50//Ohm
val Vf0     = 0.4428 //V
val rce_diodo = (1.7 - Vf0) / 50

val Vcc = 300
val Vac_rms = 28
val Vac_max_rms = 0.577 * Vcc / sqrt(2f)
val m = Vac_rms / Vac_max_rms
val f_sw = 10000
val f_red = 50

val Pmax_chip = 142
val Pmax_igbt = Pmax_chip / 6f

println("Vce0=${String.format("%.2f",Vce0)} V rce=${String.format("%.2f",rce)} Ohm Vf0=${String.format("%.2f",Vf0)} V rce_diodo=${String.format("%.3f",rce)} Ohm")

Vce0=0,90 V rce=0,01 Ohm Vf0=0,44 V rce_diodo=0,013 Ohm


![image.png](attachment:91fa90d1-00f7-4f6d-94be-8c9c92e89125.png)
![image.png](attachment:5696ebda-c833-4628-b496-db2cacdab7a6.png)

## Funciones para calcular potencia

In [19]:
fun calcular_energia_funciones_lineales(Vi: Float, Vf: Float, Ii: Float, If: Float, deltaT: Double): Double {
    val deltaV = Vf - Vi
    val deltaI = If - Ii
    return ( deltaV * deltaI / 3 + (Vi*deltaI + Ii*deltaV) / 2 + Vi * Ii ) * deltaT
}

fun calcular_potencia_encendido(il_peak: Float): Double {
    var E_ciclo = 0.0
    val tr = 0.6 * 10f.pow(-6)
    val cant_puntos = (0.5 * f_sw / f_red).roundToInt()
    for(i in (0..cant_puntos)) {
        val theta = PI / cant_puntos * i
        val Ic = il_peak * sin(theta)
        E_ciclo += calcular_energia_funciones_lineales(
            Vi = 0.9f * Vce_bloq,
            Vf = 0.1f * Vce_bloq,
            Ii = 0.1f * Ic.toFloat(),
            If = Ic.toFloat(),
            deltaT = tr
        )
    }
    return f_red * E_ciclo
}

fun calcular_potencia_apagado(il_peak: Float): Double {
    var E_ciclo = 0.0
    val tr = 1.2 * 10f.pow(-6)
    val cant_puntos = (0.5 * f_sw / f_red).roundToInt()
    for(i in (0..cant_puntos)) {
        val theta = PI / cant_puntos * i
        val Ic = il_peak * sin(theta)
        E_ciclo += calcular_energia_funciones_lineales(
            Vi = 0.1f * Vce_bloq, //TODO deberia ser Vce_sat
            Vf = 0.1f * Vce_bloq,
            Ii = 0.9f * Ic.toFloat(),
            If = 0.2f * Ic.toFloat(),
            deltaT = tr
        )
    }
    return f_red * E_ciclo
}

## Formulas para calculo de perdidas en conduccion
Se utilizan las formulas propuestas en el trabajo "Semiconductor Losses in Voltage Source and Current Source IGBT Converters Based on Analytical Derivation". Si bien trabajamos con modulacion SVPWM, las formulas en cuestion corresponden a modulacion senoidal PWM, que, segun indica el trabajo, sirven como una buena aproximacion.
### Potencia de conduccion disipada por el IGBT
Si se considera $\cos(\phi) \approx 1$ resulta:
$$ P_{C,IGBT}=\frac{(\frac{\pi}{4} + \frac{2}{3} m) rce}{2\pi} il_{peak}^2   +   \frac{(1 + \frac{\pi}{4} * m) * Vce0}{2\pi} il_{peak} $$
### Potencia de conduccion disipada por el diodo parásito
Si se considera $\cos(\phi) \approx 1$ resulta:
$$ P_{C,Diodo}=\frac{(\frac{\pi}{4} - \frac{2}{3} m) rce}{2\pi} il_{peak}^2   +   \frac{(1 - \frac{\pi}{4} * m) * Vf0}{2\pi} il_{peak} $$

In [32]:
fun calcular_potencia_conduccion_igbt(il_peak: Float): Double {
    return (PI/4 + 2/3 * m) * rce * il_peak.pow(2) / (2*PI) + (1 + PI/4 * m) * Vce0 * il_peak / (2*PI)
}

fun calcular_potencia_conduccion_diodo(il_peak: Float): Double {
    return (PI/4 - 2/3 * m) * rce_diodo * il_peak.pow(2) / (2*PI) + (1 - PI/4 * m) * Vf0 * il_peak / (2*PI)
}

# Plots

In [21]:
val corriente = List(61) { i -> i}
val p_cond_igbt     = corriente.map { i -> calcular_potencia_conduccion_igbt(i.toFloat())}
val p_cond_diodo    = corriente.map { i -> calcular_potencia_conduccion_diodo(i.toFloat())}
val p_on            = corriente.map { i -> calcular_potencia_encendido(i.toFloat())}
val p_off           = corriente.map { i -> calcular_potencia_apagado(i.toFloat())}
val p_total         = p_on
    .zip(p_off){ a,b -> a+b }
    .zip(p_cond_igbt) { a,b -> a+b }
    .zip(p_cond_diodo) { a,b -> a+b }

run {
    val data = mapOf(
        "corriente" to corriente + corriente + corriente + corriente,
        "potencia" to p_cond_igbt + p_cond_diodo + p_on + p_off,
        "legends" to List(corriente.size) { "Conduccion IGBT" } + List(corriente.size) { "Conduccion Diodo" } + List(corriente.size) { "Encendido" } + List(corriente.size) { "Apagado" }
    )  // Combine data into a map
    plot(data) { // Begin plotting
        groupBy("legends") {
            line {
                x("corriente") { axis.name = "Corriente de Pico [A]"}
                y("potencia") { axis.name = "Potencia [W]" }
                color("legends")
            }
            layout { // Set plot layout
                title = "Disipacion de potencia" // Add title
                size = 1300 to 500 // Plot dimension settings
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="VS6uN6"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"corriente",
"y":"potencia",
"color":"legends",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"potencia":[0.0,0.1705996992508716,0.34444939833596777,0.5215490972552885,0.7018987960088339,0.8854984945966038,1.0723481930185983,1.2624478912748174,1.455797589365261,1.6523972872899295,1.8522469850488221,2.0553466826419395,2.2616963800692815,2.471296077330848,2.6841457744266393,2.900245471356655,3.1195951681208953,3.3421948647193607,3.56804456115205,3.7971442574189638,4.029493953520102,4.265093649455466,4.5039433452250535,4.746043040828866,4.991392736266903,5.239992431539164,5.4918421266456505,5.746941821586361,6.0052915163612965,6.266891210970456,6.531740905413841,6.79984059969145,7.071190293803284,7.345789987749342,7.6236396815296255,7.904739375144133,8.189089068592864,8.47668876187582,8.767538454993002,9.061638147944407,9.358987840730038,9.659587533349892,9.963437225803972,10.270536918092276,10.580886610214804,10.894486302171558,11.211335993962535,11.531435685587738,11.854785377047165,12.181385068340816,12.511234759468692,12.844334450430793,13.180684141227118,13.520283831857668,13.863133522322443,14.209233212621442,14.558582902754665,14.911182592722113,15.267032282523786,15.626131972159683,15.988481661629805,0.0,0.060955063327632855,0.1281961266552657,0.20172318998289857,0.2815362533105314,0.3676353166381643,0.4600203799657971,0.55869144329343,0.6636485066210628,0.7748915699486956,0.8924206332763285,1.0162356966039614,1.1463367599315941,1.2827238232592268,1.42539688658686,1.5743559499144926,1.7296010132421256,1.8911320765697583,2.058949139897391,2.233052203225024,2.413441266552657,2.6001163298802896,2.7930773932079225,2.992324456535555,3.197857519863188,3.409676583190821,3.6277816465184536,3.8521727098460863,4.08284977317372,4.319812836501352,4.563061899828985,4.812596963156618,5.068418026484251,5.330525089811884,5.5989181531395165,5.873597216467148,6.154562279794781,6.441813343122415,6.735350406450047,7.03517346977768,7.341282533105312,7.653677596432947,7.9723586597605784,8.297325723088212,8.628578786415845,8.966117849743476,9.30994291307111,9.660053976398743,10.016451039726375,10.379134103054009,10.748103166381641,11.123358229709275,11.504899293036905,11.89272635636454,12.286839419692171,12.687238483019806,13.093923546347439,13.506894609675072,13.926151673002703,14.351694736330337,14.783523799657969,0.0,0.12317579521551555,0.2463515904310311,0.3695273904530651,0.4927031808620622,0.6158789801116201,0.7390547809061302,0.8622305689404783,0.9854063617241244,1.108582175278797,1.2317579602232402,1.3549337244538768,1.4781095618122604,1.6012853235251974,1.7244611378809567,1.8476369095502523,1.9708127234482489,2.0939884574664838,2.217164350557594,2.3403401450006327,2.4635159204464805,2.586691599304192,2.7098674489077537,2.8330432907293335,2.9562191236245208,3.0793948473430732,3.2025706470503947,3.3257464373735597,3.4489222757619133,3.5720980642540234,3.6952738191005046,3.818449632082974,3.9416254468964977,4.0648011408604106,4.1879769149329675,4.311152867075592,4.434328701115188,4.557504304441894,4.680680290001265,4.803856037523527,4.927031840892961,5.050207640142518,5.173383198608384,5.296559109552278,5.419734897815507,5.542910883832643,5.666086581458667,5.789262291902071

In [22]:
run {
    val data = mapOf(
        "corriente" to corriente,
        "potencia" to p_total,
    )  // Combine data into a map

    plot(data) { // Begin plotting
        hLine {
            yIntercept.constant(Pmax_igbt)
            color = Color.RED
            type = LineType.DASHED
        }
        line {
            x("corriente") { axis.name = "Corriente de Pico [A]"}
            y("potencia") { axis.name = "Potencia [W]" }
            color = Color.YELLOW
        }
        layout { // Set plot layout
            title = "Disipacion de potencia" // Add title
            size = 1300 to 500 // Plot dimension settings
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Ls1JJJ"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"Disipacion de potencia"
},
"mapping":{
},
"data":{
"potencia":[0.0,0.4177507296920611,0.8450374592183468,1.2818601950018538,1.728218917773592,2.1841136553427054,2.6495443885117287,3.124511113389672,3.609013832894777,4.1030525858797375,4.606627306541025,5.119737996251929,5.642384771055542,6.174567468331318,6.716286218656348,7.267540916172777,7.828331655179928,8.398658324212345,8.978521158331667,9.567919871321164,10.166854596504507,10.775325226078348,11.393331972445033,12.020874748858343,12.657953518239424,13.304568177019243,13.96071890864659,14.626405620538769,15.301628404820715,15.986387118670155,16.680681795046084,17.384512528133378,18.09787926792135,18.820781917135218,19.553220600515594,20.2951954926993,21.046706262952096,21.807752806903864,22.5783356827974,23.358454349784928,24.14810912669885,24.947299819676235,25.756026379049736,26.574289142877824,27.40208786465476,28.23942280049932,29.08629341002149,29.942700108641798,30.808642940992208,31.68412149027889,32.56913625042885,33.463687067633494,34.367773705229,35.28139635593388,36.204555120227255,37.137249968583376,38.0794806796735,39.031247286685485,39.992550097923186,40.963388567503415,41.943763440894294],
"corriente":[0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,51.0,52.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0]
},
"ggsize":{
"width":1300.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Corriente de Pico [A]",
"limits":[null,null]
},{
"aesthetic":"y",
"name":"Potencia [W]",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"yintercept":23.66666603088379,
"color":"#ee6666",
"linetype":"dashed",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"hline",
"data":{
}
},{
"mapping":{
"x":"corriente",
"y":"potencia"
},
"stat":"identity",
"color":"#fac858",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"int",
"column":"corriente"
},{
"type":"float",
"column":"potencia"
}]
},
"spec_id":"11"
};
 var containerDiv = document.getElementById("Ls1JJJ");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 1300.0,
 height: 500.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 <path d="M56.456308594957434 401.8636363636364 L56.456308594957434 401.8636363636364 L75.27507812660991 398.05175625707 L94.09384765826239 394.1528623274731 L112.91261718991487 390.1669545162373 L131.73138672156733 386.09403299918773 L150.55015625321982 381.93409752257224 L169.36892578487232 377.68714826156344 L188.18769531652475 373.3531852516656 L207.00646484817725 368.93220846625053 L225.82523437982974 364.4242175507962 L244.64400391148223 359.8292131057448 L263.46277344313467 355.14719511856543 L282.28154297478716 350.3781625303913 L301.10031250643965 345.5221168250978 L319.9190820380921 340.57905681119877 L338.7378515697446 335.54898345462294 L357.55662110139707 330.43189589491055 L376.37539063304956 325.2277951491585 L395.19416016470205 319.9366790724829 L414.012929696

In [23]:
println("Potencia activa maxima: ${3*28*40/sqrt(2f)} W")

Potencia activa maxima: 2375.879 W
